# RoadLens C1: Base Model Training on Google Colab

This notebook guides you through training the **YOLOv8m 4-class road damage detector (B0 baseline)** on Google Colab with a free GPU (T4).

### Checklist before running:
1. **Enable GPU**: Click `Runtime` in the top menu -> `Change runtime type` -> select **T4 GPU** -> click `Save`.
2. Run the cells sequentially by clicking the **Play button** on the left of each cell (or press `Shift + Enter`).

In [ ]:
# Step 1: Verify GPU Acceleration
!nvidia-smi

### Step 2: Mount Google Drive
Mounting Google Drive ensures your trained weights (`best.pt`) and training curves are automatically saved permanently to your personal Google Drive so you never lose progress if the Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 3: Clone RoadLens Repository & Install Dependencies
Clone your active branch and install `ultralytics`.

In [ ]:
# Clone the exact feature branch
!git clone -b feature/c1-base-model-training https://github.com/dinukadilshan03/RoadLens.git
%cd RoadLens

# Install YOLO dependencies
!pip install --upgrade ultralytics pyyaml pandas

### Step 4: Dataset Preparation
If you uploaded your `rdd_cleaned.zip` to your Google Drive, unzip it directly into `ml/c1/data/`.

> Note: Update the path below if your zip file is stored in a specific folder in your Google Drive.

In [ ]:
import os
import shutil

zip_drive_path = '/content/drive/MyDrive/rdd_cleaned.zip'
target_dir = 'ml/c1/data/rdd_cleaned'

if os.path.exists(zip_drive_path):
    print(f"Extracting dataset from {zip_drive_path}...")
    !unzip -q {zip_drive_path} -d ml/c1/data/
    print("Dataset extraction complete!")
else:
    print("Please upload rdd_cleaned.zip to your Google Drive root or update zip_drive_path above.")

# Quick sanity check of dataset structure
!ls -la ml/c1/data/rdd_cleaned

### Step 5: Launch Base Model Training
Launch training with the road distress hyperparameter configuration (`ml/c1/configs/hyp_base.yaml`).
- Architecture: `yolov8m.pt`
- Image Resolution: `800x800`
- Epochs: `50` (with early stopping patience = 15)
- Device: `0` (CUDA GPU)

In [ ]:
!python ml/c1/scripts/base_model_training.py \
    --data ml/c1/configs/rdd2022_4class.yaml \
    --hyp ml/c1/configs/hyp_base.yaml \
    --model yolov8m.pt \
    --epochs 50 \
    --imgsz 800 \
    --batch 16 \
    --device 0

### Step 6: Backup Checkpoints and Run Metrics to Google Drive
Once training completes, this cell copies the best checkpoint weights (`base_model_rdd2022.pt`) and validation plots (`confusion_matrix.png`, `results.png`, etc.) directly into a dedicated folder on your Google Drive.

In [ ]:
import os
import shutil

drive_backup_dir = "/content/drive/MyDrive/RoadLens_C1_Weights"
os.makedirs(drive_backup_dir, exist_ok=True)

best_weights = "ml/c1/weights/base_model_rdd2022.pt"
if os.path.exists(best_weights):
    shutil.copy2(best_weights, os.path.join(drive_backup_dir, "base_model_rdd2022.pt"))
    print(f"Best model checkpoint saved to: {drive_backup_dir}/base_model_rdd2022.pt")

# Archive complete training run plots and logs
!tar -czf /content/drive/MyDrive/RoadLens_C1_Weights/training_run_logs.tar.gz ml/c1/runs/train/
print("Training logs and curves backed up to Google Drive successfully!")